# ONG v3 — Pilot: EfficientNetV2-L (recipe bersih, prior-respecting)

Re-run **fair** untuk EfficientNetV2-L dengan recipe yang sama seperti pilot DINOv2 —
`--sampler-power 0 --loss ce` (tanpa double-balancing; menghormati prior Bulbophyllum/
Dendrobium). Tujuannya menjawab apakah hasil bake-off lama yang rendah (global 59.1% /
macro 49.9%) itu **artefak recipe yang overfit**, bukan keterbatasan arsitektur — sama
seperti DINOv2 yang melonjak 66.8% → 88.9% global setelah recipe bersih.

Notebook ini bisa dipakai untuk **dua backbone**: ganti `MODEL` di Sel 2
(`'effnetv2l'` atau `'convnextv2l'`). Menyimpan **`best_model_macro.pth` + `best_model_global.pth`**,
meng-eval **keduanya** pada test split frozen, lalu mencetak tabel banding.

**Baseline & pembanding (test split frozen yang SAMA, ~57–58 genus):**

| Model | global T1 | macro T1 | species R@5 | genus R@5 |
|---|---|---|---|---|
| effb4 baseline (OPTIMISTIC, ada overlap) | 90.1 | **74.0** | 73.4 | 94.1 |
| DINOv2-L (pilot, recipe bersih) | 88.9 | 66.9 | 86.6 | **98.7** |
| effnetv2-L (bake-off lama, overfit) | 59.1 | 49.9 | 83.0 | 97.4 |
| convnextv2-L (bake-off lama, overfit) | 66.4 | 53.6 | 83.5 | 96.2 |

**Sebelum run**, upload ke Drive `MyDrive/ONG_v3/`:
`scripts/03_train_bakeoff_colab.py`, `scripts/13_evaluate.py`,
`data/splits/{train,val,test}_live.csv`, `data/splits/all_images.csv`, `photos.zip`.
Pakai L4 (Runtime → Change runtime type → L4), lalu **Run all**. ±15–20 compute unit / backbone.

In [ ]:
# Sel 1 - Mount Drive, install deps, cek GPU
from google.colab import drive; drive.mount('/content/drive')
!pip -q install -U timm open_clip_torch faiss-cpu
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > L4, lalu Run all lagi.'

In [ ]:
# Sel 2 - Pilih backbone + set ROOT, verifikasi file di Drive
import os
ROOT = '/content/drive/MyDrive/ONG_v3'

# img-size HARUS = registry di 03_train_bakeoff_colab.py supaya train==eval konsisten
PILOT = {
    'effnetv2l':   ('tf_efficientnetv2_l.in21k_ft_in1k',    448),
    'convnextv2l': ('convnextv2_large.fcmae_ft_in22k_in1k', 384),
}
MODEL = 'effnetv2l'                 # <- ganti ke 'convnextv2l' utk re-run ConvNeXt
EVAL_NAME, IMG = PILOT[MODEL]
print(f'Backbone: {MODEL}  ({EVAL_NAME})  img={IMG}')

need = ['scripts/03_train_bakeoff_colab.py', 'scripts/13_evaluate.py',
        'data/splits/train_live.csv', 'data/splits/val_live.csv',
        'data/splits/test_live.csv', 'data/splits/all_images.csv', 'photos.zip']
status = {f: os.path.exists(f'{ROOT}/{f}') for f in need}
print('present:', status)
missing = [f for f, ok in status.items() if not ok]
assert not missing, f'Missing di {ROOT}/: {missing} - upload dulu, lalu Run all lagi.'
print('Semua file siap.')

In [ ]:
# Sel 3 - Unzip photos ke disk lokal cepat (/content/photos/{Genus}/*.jpg)
!unzip -q -o "{ROOT}/photos.zip" -d /content/
nested = '/content/photos/photos'
if os.path.isdir(nested):
    import shutil
    for it in os.listdir(nested):
        shutil.move(os.path.join(nested, it), '/content/photos')
    os.rmdir(nested)
assert os.path.isdir('/content/photos'), 'Tidak ada /content/photos setelah unzip - cek layout zip.'
print('folder genus:', len(os.listdir('/content/photos')))

In [ ]:
# Sel 4 - Train (recipe bersih) + eval KEDUA checkpoint pada test split frozen
# --img-size diteruskan ke train DAN eval supaya tidak bisa drift.
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model {MODEL} --drive-root {ROOT} \
    --photos-root /content/photos --img-size {IMG} --sampler-power 0 --loss ce

for tag in ['macro', 'global']:
    ckpt = f'{ROOT}/models/{MODEL}/best_model_{tag}.pth'
    out  = f'{ROOT}/eval/{MODEL}' if tag == 'macro' else f'{ROOT}/eval/{MODEL}_{tag}'
    assert os.path.exists(ckpt), f'Checkpoint hilang: {ckpt} - training gagal (lihat log di atas).'
    print(f'\n{"="*70}\n[EVAL] {MODEL} [{tag}-selected]\n{"="*70}')
    get_ipython().system(
        f'python {ROOT}/scripts/13_evaluate.py --model {EVAL_NAME} --img-size {IMG} '
        f'--checkpoint {ckpt} --vocab {ROOT}/models/{MODEL}/vocab.json '
        f'--splits-dir {ROOT}/data/splits --photos-root /content/photos --out-dir {out}'
    )

In [ ]:
# Sel 5 - Tabel banding (baca setiap eval/<dir>/results.json) + simpan ringkasan markdown
import json, datetime
from pathlib import Path

BASE = dict(name='baseline effb4 (FROZEN)', macro_top1=74.0, macro_top5=84.8,
            global_top1=90.1, sp_r5=73.4, gn_r5=94.1, ece=0.082)
PRETTY = {'bioclip2': 'BioCLIP-2 ViT-L/14', 'dinov2l': 'DINOv2 ViT-L/14',
          'convnextv2l': 'ConvNeXt-V2-L', 'effnetv2l': 'EfficientNetV2-L'}

def pretty(name):
    suf = ''
    if name.endswith('_global'):
        name, suf = name[:-7], ' [global-sel]'
    return PRETTY.get(name, name) + suf

def read(p):
    r = json.loads(Path(p).read_text())
    c = r.get('classification', {}) or {}
    q = r.get('retrieval', {}) or {}
    pc = lambda d, k: round(d[k] * 100, 1) if d.get(k) is not None else None
    return dict(macro_top1=pc(c, 'macro_top1'), macro_top5=pc(c, 'macro_top5'),
                global_top1=pc(c, 'global_top1'), sp_r5=pc(q, 'species_recall@5'),
                gn_r5=pc(q, 'genus_recall@5'),
                ece=(round(c['ece'], 3) if c.get('ece') is not None else None))

evdir = Path(ROOT) / 'eval'
found = sorted(p.parent.name for p in evdir.glob('*/results.json')
               if not p.parent.name.startswith('_'))
f2 = lambda x: '-' if x is None else f'{x}'
date = datetime.date.today().isoformat()
L = [f'## {date} - Pilot {MODEL} (recipe bersih; protokol split frozen)',
     '',
     'Baseline = effb4 FROZEN (OPTIMISTIC: train/test overlap; macro>=5 76.2%). '
     '`[global-sel]` = checkpoint dipilih by val global top-1.',
     '',
     '| Model | macro T1 | macro T5 | global T1 | species R@5 | genus R@5 | ECE |',
     '|-------|----------|----------|-----------|-------------|-----------|-----|',
     f"| {BASE['name']} | **{BASE['macro_top1']}** | {BASE['macro_top5']} | {BASE['global_top1']} | {BASE['sp_r5']} | {BASE['gn_r5']} | {BASE['ece']} |"]
best_cls = best_ret = None
for name in found:
    r = read(evdir / name / 'results.json')
    L.append(f"| {pretty(name)} | **{f2(r['macro_top1'])}** | {f2(r['macro_top5'])} | "
             f"{f2(r['global_top1'])} | {f2(r['sp_r5'])} | {f2(r['gn_r5'])} | {f2(r['ece'])} |")
    if r['macro_top1'] is not None and (best_cls is None or r['macro_top1'] > best_cls[1]):
        best_cls = (name, r['macro_top1'])
    if r['sp_r5'] is not None and (best_ret is None or r['sp_r5'] > best_ret[1]):
        best_ret = (name, r['sp_r5'])
L.append('')
if best_cls:
    d = round(best_cls[1] - BASE['macro_top1'], 1)
    L.append(f"- **Best genus (macro top-1):** {pretty(best_cls[0])} = {best_cls[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
if best_ret:
    d = round(best_ret[1] - BASE['sp_r5'], 1)
    L.append(f"- **Best retrieval (species R@5):** {pretty(best_ret[0])} = {best_ret[1]}% "
             f"({'+' if d >= 0 else ''}{d} vs baseline).")
snippet = '\n'.join(L)
out = evdir / f'pilot_{MODEL}_summary_{date}.md'
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(snippet, encoding='utf-8')
print(snippet)
print(f'\nSaved -> {out}  (download via Files panel; tempel ke ONGv3_progress.Rmd)')